# 08. Application + все дополнительные признаки

## Цель

Кратко:
- объединить пять сохранённых таблиц признаков с `application_train`;
- использовать единый client split;
- поддержать полный GPU, полный CPU и облегчённый CPU-debug режимы;
- обучить модель и сохранить метрики без смешивания debug-артефактов
  с полноценными результатами.


## 1. Импорты и пути


In [1]:
from pathlib import Path
import sys


def _is_project_root(path):
    return (
        (path / "src").is_dir()
        and (path / "notebooks").is_dir()
        and (path / "data").is_dir()
    )


project_candidates = [
    Path.cwd(),
    *Path.cwd().parents,
    Path("/content/credit-scoring-system"),
]

if "google.colab" in sys.modules:
    from google.colab import drive

    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.is_dir():
        drive.mount("/content/drive")

    default_drive_project = (
        drive_root / "credit-scoring-system"
    )
    project_candidates.append(default_drive_project)

    if not any(
        _is_project_root(path)
        for path in project_candidates
    ):
        project_candidates.extend(
            config_path.parents[1]
            for config_path in drive_root.rglob("src/config.py")
        )

PROJECT_ROOT = next(
    (
        path.resolve()
        for path in project_candidates
        if _is_project_root(path)
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не найден корень credit-scoring-system. На Google Drive "
        "должна находиться вся папка проекта с src/, notebooks/ и data/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_setup import setup_notebook


PROJECT_ROOT = setup_notebook()

Mounted at /content/drive
Installing missing dependency: catboost
Environment: Google Colab
Python: 3.12.13
Project root: /content/drive/MyDrive/credit-scoring-system
Raw data: /content/drive/MyDrive/credit-scoring-system/data/raw
Models: /content/drive/MyDrive/credit-scoring-system/models
Reports: /content/drive/MyDrive/credit-scoring-system/reports


In [2]:
import json
import os
import platform
from datetime import datetime, timezone

import catboost
import numpy as np
import pandas as pd
from catboost import (
    CatBoostClassifier,
    Pool,
    cv as catboost_cv,
)
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
)

from src.config import (
    IN_COLAB,
    INTERIM_DATA_DIR as DATA_INTERIM_DIR,
    MODELS_DIR,
    PROCESSED_DATA_DIR as DATA_PROCESSED_DIR,
    find_data_file,
)
from src.experiment_tracking import save_experiment_result
from src.model_config import get_catboost_gpu_count


APPLICATION_PATH = find_data_file("application_train.csv")
CLIENT_SPLIT_PATH = DATA_PROCESSED_DIR / "client_split.csv"
BUREAU_FEATURES_PATH = DATA_INTERIM_DIR / "bureau_features.csv"
PREVIOUS_FEATURES_PATH = (
    DATA_INTERIM_DIR / "previous_application_features.csv"
)
INSTALLMENTS_FEATURES_PATH = (
    DATA_INTERIM_DIR / "installments_features.csv"
)
POS_CASH_FEATURES_PATH = DATA_INTERIM_DIR / "pos_cash_features.csv"
CREDIT_CARD_FEATURES_PATH = (
    DATA_INTERIM_DIR / "credit_card_features.csv"
)

required_feature_paths = [
    BUREAU_FEATURES_PATH,
    PREVIOUS_FEATURES_PATH,
    INSTALLMENTS_FEATURES_PATH,
    POS_CASH_FEATURES_PATH,
    CREDIT_CARD_FEATURES_PATH,
]
missing_feature_paths = [
    path
    for path in required_feature_paths
    if not path.exists()
]
if missing_feature_paths:
    raise FileNotFoundError(
        "Для запуска 08 сначала выполните ноутбуки 03–07. "
        "Не найдены: "
        + ", ".join(str(path) for path in missing_feature_paths)
    )

RANDOM_STATE = 42


## 2. Конфигурация режима запуска


In [3]:
REQUESTED_RUN_MODE = "auto"

valid_run_modes = {
    "auto",
    "full_gpu",
    "full_cpu",
    "cpu_debug",
}
if REQUESTED_RUN_MODE not in valid_run_modes:
    raise ValueError(
        "REQUESTED_RUN_MODE должен быть одним из: "
        + ", ".join(sorted(valid_run_modes))
    )

gpu_count = get_catboost_gpu_count()
RUN_ENVIRONMENT = "Google Colab" if IN_COLAB else "local"
CPU_THREAD_COUNT = max(1, int(os.cpu_count() or 1))

if REQUESTED_RUN_MODE == "auto":
    ACTUAL_RUN_MODE = (
        "full_gpu"
        if gpu_count > 0
        else "cpu_debug"
    )
else:
    ACTUAL_RUN_MODE = REQUESTED_RUN_MODE

if ACTUAL_RUN_MODE == "full_gpu":
    if gpu_count < 1:
        raise RuntimeError(
            "Режим full_gpu запрошен, но CatBoost не обнаружил GPU. "
            "В Google Colab выберите Runtime → Change runtime type "
            "→ T4 GPU либо установите REQUESTED_RUN_MODE='auto'."
        )
    catboost_device_config = {
        "task_type": "GPU",
        "devices": "0",
    }
else:
    catboost_device_config = {
        "task_type": "CPU",
        "thread_count": -1,
    }

IS_DEBUG = ACTUAL_RUN_MODE == "cpu_debug"

if IS_DEBUG:
    CV_FOLDS = 2
    MAX_ITERATIONS = 150
    EARLY_STOPPING_ROUNDS = 30
    EXPERIMENT = "application_all_features_cpu_debug"
    MODEL_NAME = "catboost_all_features_cpu_debug"
    print("Внимание: используется облегчённый CPU-debug режим.")
    print(
        "Полученные метрики не являются финальными метриками проекта."
    )
else:
    CV_FOLDS = 3
    MAX_ITERATIONS = 1000
    EARLY_STOPPING_ROUNDS = 100
    EXPERIMENT = "application_all_features"
    MODEL_NAME = "catboost_all_features"

if ACTUAL_RUN_MODE == "full_cpu":
    print("Используется полный CPU-режим.")
    print(
        "Кросс-валидация и обучение могут выполняться долго."
    )

print("Запрошенный режим:", REQUESTED_RUN_MODE)
print("Фактический режим:", ACTUAL_RUN_MODE)
print("Среда запуска:", RUN_ENVIRONMENT)
print("Количество найденных GPU:", gpu_count)
print(
    "Устройство CatBoost:",
    catboost_device_config["task_type"],
)


Внимание: используется облегчённый CPU-debug режим.
Полученные метрики не являются финальными метриками проекта.
Запрошенный режим: auto
Фактический режим: cpu_debug
Среда запуска: Google Colab
Количество найденных GPU: 0
Устройство CatBoost: CPU


## 3. Загрузка application_train


In [4]:
application = pd.read_csv(APPLICATION_PATH)

if "DAYS_EMPLOYED" in application.columns:
    application["DAYS_EMPLOYED"] = application[
        "DAYS_EMPLOYED"
    ].replace(365243, np.nan)

assert application["SK_ID_CURR"].is_unique
assert application["TARGET"].isin([0, 1]).all()

print("Application:", application.shape)
display(application.head())


Application: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Загрузка сохранённых таблиц признаков


In [5]:
bureau_features = pd.read_csv(BUREAU_FEATURES_PATH)
previous_features = pd.read_csv(PREVIOUS_FEATURES_PATH)
installments_features = pd.read_csv(INSTALLMENTS_FEATURES_PATH)
pos_cash_features = pd.read_csv(POS_CASH_FEATURES_PATH)
credit_card_features = pd.read_csv(CREDIT_CARD_FEATURES_PATH)

assert bureau_features["SK_ID_CURR"].is_unique
assert previous_features["SK_ID_CURR"].is_unique
assert installments_features["SK_ID_CURR"].is_unique
assert pos_cash_features["SK_ID_CURR"].is_unique
assert credit_card_features["SK_ID_CURR"].is_unique

assert "TARGET" not in bureau_features.columns
assert "TARGET" not in previous_features.columns
assert "TARGET" not in installments_features.columns
assert "TARGET" not in pos_cash_features.columns
assert "TARGET" not in credit_card_features.columns


## 5. Добавление признаков bureau


In [6]:
full_data = application.copy()

full_data = full_data.merge(
    bureau_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one",
)

assert len(full_data) == len(application)
assert full_data["SK_ID_CURR"].is_unique


## 6. Добавление признаков previous_application


In [7]:
full_data = full_data.merge(
    previous_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one",
)

assert len(full_data) == len(application)
assert full_data["SK_ID_CURR"].is_unique


## 7. Добавление признаков installments_payments


In [8]:
full_data = full_data.merge(
    installments_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one",
)

assert len(full_data) == len(application)
assert full_data["SK_ID_CURR"].is_unique


## 8. Добавление признаков POS_CASH_balance


In [9]:
full_data = full_data.merge(
    pos_cash_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one",
)

assert len(full_data) == len(application)
assert full_data["SK_ID_CURR"].is_unique


## 9. Добавление признаков credit_card_balance


In [10]:
full_data = full_data.merge(
    credit_card_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one",
)

assert len(full_data) == len(application)
assert full_data["SK_ID_CURR"].is_unique

print("Application:", application.shape)
print("Все признаки:", full_data.shape)
print(
    "Добавлено признаков:",
    full_data.shape[1] - application.shape[1],
)


Application: (307511, 122)
Все признаки: (307511, 174)
Добавлено признаков: 52


## 10. Чтение единого client split


In [11]:
if not CLIENT_SPLIT_PATH.exists():
    raise FileNotFoundError(
        "Сначала выполните notebooks/02_application_baseline.ipynb. "
        f"Ожидаемый файл: {CLIENT_SPLIT_PATH}"
    )

client_split = pd.read_csv(CLIENT_SPLIT_PATH)

assert client_split.columns.tolist() == ["SK_ID_CURR", "split"]
assert client_split["SK_ID_CURR"].is_unique
assert set(client_split["split"]) == {"train", "holdout"}
assert set(client_split["SK_ID_CURR"]) == set(application["SK_ID_CURR"])

modeling_data = full_data.merge(
    client_split,
    on="SK_ID_CURR",
    how="inner",
    validate="one_to_one",
)

assert len(modeling_data) == len(application)
assert modeling_data["SK_ID_CURR"].is_unique

print(client_split["split"].value_counts())


split
train      246008
holdout     61503
Name: count, dtype: int64


## 11. Формирование train и holdout

Сначала используются все клиенты из сохранённого split. Только в
`cpu_debug` обе части уменьшаются стратифицированно. `TARGET`,
идентификатор клиента и техническая колонка split не передаются модели.


In [12]:
train_data = modeling_data[
    modeling_data["split"].eq("train")
].copy()

holdout_data = modeling_data[
    modeling_data["split"].eq("holdout")
].copy()

train_target_rate_before = float(train_data["TARGET"].mean())
holdout_target_rate_before = float(holdout_data["TARGET"].mean())

if IS_DEBUG and len(train_data) > 50_000:
    train_data, _ = train_test_split(
        train_data,
        train_size=50_000,
        stratify=train_data["TARGET"],
        random_state=RANDOM_STATE,
    )

if IS_DEBUG and len(holdout_data) > 15_000:
    holdout_data, _ = train_test_split(
        holdout_data,
        train_size=15_000,
        stratify=holdout_data["TARGET"],
        random_state=RANDOM_STATE,
    )

train_data = train_data.reset_index(drop=True)
holdout_data = holdout_data.reset_index(drop=True)

train_target_rate_after = float(train_data["TARGET"].mean())
holdout_target_rate_after = float(holdout_data["TARGET"].mean())

assert abs(
    train_target_rate_after - train_target_rate_before
) <= 0.01
assert abs(
    holdout_target_rate_after - holdout_target_rate_before
) <= 0.01

feature_columns = [
    column
    for column in modeling_data.columns
    if column not in {
        "TARGET",
        "SK_ID_CURR",
        "split",
    }
]

X_train = train_data[feature_columns]
y_train = train_data["TARGET"].astype(int)

X_holdout = holdout_data[feature_columns]
y_holdout = holdout_data["TARGET"].astype(int)

assert "TARGET" not in X_train.columns
assert "SK_ID_CURR" not in X_train.columns
assert "split" not in X_train.columns

print("Запрошенный режим:", REQUESTED_RUN_MODE)
print("Фактический режим:", ACTUAL_RUN_MODE)
print("Среда запуска:", RUN_ENVIRONMENT)
print("Количество найденных GPU:", gpu_count)
print(
    "Устройство CatBoost:",
    catboost_device_config["task_type"],
)
print("Количество train-строк:", len(train_data))
print("Количество holdout-строк:", len(holdout_data))
print("Количество признаков:", len(feature_columns))
print("Количество CV-фолдов:", CV_FOLDS)
print(
    "Максимальное количество итераций:",
    MAX_ITERATIONS,
)
print(
    "Доля TARGET=1 train до/после:",
    f"{train_target_rate_before:.4f}",
    f"{train_target_rate_after:.4f}",
)
print(
    "Доля TARGET=1 holdout до/после:",
    f"{holdout_target_rate_before:.4f}",
    f"{holdout_target_rate_after:.4f}",
)


Запрошенный режим: auto
Фактический режим: cpu_debug
Среда запуска: Google Colab
Количество найденных GPU: 0
Устройство CatBoost: CPU
Количество train-строк: 50000
Количество holdout-строк: 15000
Количество признаков: 172
Количество CV-фолдов: 2
Максимальное количество итераций: 150
Доля TARGET=1 train до/после: 0.0807 0.0807
Доля TARGET=1 holdout до/после: 0.0807 0.0807


## 12. Подготовка данных для CatBoost

Категориальные пропуски заменяются строкой. Числовые `NaN` остаются без
изменений: CatBoost обрабатывает их самостоятельно.


In [13]:
categorical_columns = (
    X_train
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

X_train_catboost = X_train.copy()
X_holdout_catboost = X_holdout.copy()

X_train_catboost[categorical_columns] = (
    X_train_catboost[categorical_columns]
    .fillna("Unknown")
    .astype(str)
)

X_holdout_catboost[categorical_columns] = (
    X_holdout_catboost[categorical_columns]
    .fillna("Unknown")
    .astype(str)
)

print("Категориальных признаков:", len(categorical_columns))


Категориальных признаков: 16


## 13. Стратифицированная кросс-валидация


In [14]:
cv_splitter = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


## 14. Pool для CatBoost


In [15]:
train_pool = Pool(
    data=X_train_catboost,
    label=y_train,
    cat_features=categorical_columns,
)


## 15. Параметры CatBoost


In [16]:
catboost_params = {
    "iterations": MAX_ITERATIONS,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "custom_metric": ["PRAUC:type=Classic"],
    "auto_class_weights": "Balanced",
    "random_seed": RANDOM_STATE,
    "allow_writing_files": False,
    "verbose": False,
    **catboost_device_config,
}

# catboost.cv 1.2.10 не принимает thread_count=-1 в params.
# Явное число доступных ядер эквивалентно использованию всех CPU.
if catboost_params["task_type"] == "CPU":
    catboost_params["thread_count"] = CPU_THREAD_COUNT


## 16. Библиотечная CV CatBoost

CV выполняется только на выбранной train-части. OOF-предсказания не
требуются, поэтому используется `catboost.cv()` без ручного цикла.


In [17]:
catboost_cv_results = catboost_cv(
    pool=train_pool,
    params=catboost_params,
    folds=cv_splitter,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    as_pandas=True,
    verbose=100,
)


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:877: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


Training on fold [0/2]
0:	test: 0.6765864	best: 0.6765864 (0)	total: 149ms	remaining: 22.3s
100:	test: 0.7487767	best: 0.7492083 (98)	total: 12.5s	remaining: 6.08s
149:	test: 0.7519581	best: 0.7519658 (148)	total: 17.1s	remaining: 0us

bestTest = 0.7519657518
bestIteration = 148

Training on fold [1/2]
0:	test: 0.6901523	best: 0.6901523 (0)	total: 104ms	remaining: 15.5s
100:	test: 0.7602922	best: 0.7602922 (100)	total: 12.2s	remaining: 5.9s
149:	test: 0.7652595	best: 0.7652595 (149)	total: 16.7s	remaining: 0us

bestTest = 0.7652595184
bestIteration = 149



## 17. Лучшая итерация и CV-метрики


In [18]:
auc_mean_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-AUC")
    and column.endswith("-mean")
)

auc_std_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-AUC")
    and column.endswith("-std")
)

pr_auc_mean_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-PRAUC")
    and column.endswith("-mean")
)

pr_auc_std_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-PRAUC")
    and column.endswith("-std")
)

best_cv_index = catboost_cv_results[
    auc_mean_column
].idxmax()

best_cv_row = catboost_cv_results.loc[
    best_cv_index
]

best_iteration = int(
    best_cv_row["iterations"]
) + 1

cv_roc_auc = float(
    best_cv_row[auc_mean_column]
)

cv_roc_auc_std = float(
    best_cv_row[auc_std_column]
)

cv_pr_auc = float(
    best_cv_row[pr_auc_mean_column]
)

cv_pr_auc_std = float(
    best_cv_row[pr_auc_std_column]
)

print(f"Лучшая итерация: {best_iteration}")
print(
    f"CV ROC-AUC: {cv_roc_auc:.4f} "
    f"± {cv_roc_auc_std:.4f}"
)
print(
    f"CV PR-AUC: {cv_pr_auc:.4f} "
    f"± {cv_pr_auc_std:.4f}"
)


Лучшая итерация: 150
CV ROC-AUC: 0.7586 ± 0.0094
CV PR-AUC: nan ± nan


## 18. Итоговая модель CatBoost


In [19]:
catboost_model = CatBoostClassifier(
    iterations=best_iteration,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    allow_writing_files=False,
    verbose=100,
    **catboost_device_config,
)


## 19. Обучение итоговой модели


In [20]:
catboost_model.fit(
    X_train_catboost,
    y_train,
    cat_features=categorical_columns,
)


0:	total: 133ms	remaining: 19.8s
100:	total: 15.2s	remaining: 7.38s
149:	total: 23.7s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, auto_class_weights='Balanced', depth=6, eval_metric='AUC', iterations=150, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='CPU', verbose=100)

## 20. Вероятности на holdout


In [21]:
holdout_proba = catboost_model.predict_proba(
    X_holdout_catboost
)[:, 1]


## 21. Holdout-метрики


In [22]:
holdout_roc_auc = roc_auc_score(
    y_holdout,
    holdout_proba,
)

holdout_pr_auc = average_precision_score(
    y_holdout,
    holdout_proba,
)

print(f"Holdout ROC-AUC: {holdout_roc_auc:.4f}")
print(f"Holdout PR-AUC: {holdout_pr_auc:.4f}")


Holdout ROC-AUC: 0.7655
Holdout PR-AUC: 0.2498


## 22. Сохранение модели и metadata


In [23]:
if IS_DEBUG:
    model_path = (
        MODELS_DIR / "catboost_all_features_cpu_debug.cbm"
    )
    metadata_path = (
        MODELS_DIR
        / "catboost_all_features_cpu_debug_metadata.json"
    )
else:
    model_path = MODELS_DIR / "catboost_all_features.cbm"
    metadata_path = (
        MODELS_DIR / "catboost_all_features_metadata.json"
    )

catboost_model.save_model(str(model_path))

metadata_parameters = {
    "iterations": int(best_iteration),
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "auto_class_weights": "Balanced",
    "random_seed": int(RANDOM_STATE),
    "allow_writing_files": False,
    "verbose": 100,
}
metadata_parameters.update(catboost_device_config)

model_metadata = {
    "model_name": MODEL_NAME,
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "catboost_version": str(catboost.__version__),
    "python_version": platform.python_version(),
    "run_mode": ACTUAL_RUN_MODE,
    "is_debug": bool(IS_DEBUG),
    "device": catboost_device_config["task_type"],
    "gpu_count": int(gpu_count),
    "random_state": int(RANDOM_STATE),
    "cv_folds": int(CV_FOLDS),
    "max_iterations": int(MAX_ITERATIONS),
    "early_stopping_rounds": int(EARLY_STOPPING_ROUNDS),
    "best_iteration": int(best_iteration),
    "parameters": metadata_parameters,
    "feature_names": [str(column) for column in feature_columns],
    "categorical_features": [
        str(column)
        for column in categorical_columns
    ],
    "source_tables": [
        "application_train.csv",
        "bureau.csv",
        "bureau_balance.csv",
        "previous_application.csv",
        "installments_payments.csv",
        "POS_CASH_balance.csv",
        "credit_card_balance.csv",
    ],
    "n_train": int(len(train_data)),
    "n_holdout": int(len(holdout_data)),
    "n_features": int(X_train.shape[1]),
    "cv_roc_auc": float(cv_roc_auc),
    "cv_roc_auc_std": float(cv_roc_auc_std),
    "cv_pr_auc": float(cv_pr_auc),
    "cv_pr_auc_std": float(cv_pr_auc_std),
    "holdout_roc_auc": float(holdout_roc_auc),
    "holdout_pr_auc": float(holdout_pr_auc),
}

with metadata_path.open(
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        model_metadata,
        metadata_file,
        ensure_ascii=False,
        indent=2,
    )

print("Модель сохранена:", model_path)
print("Metadata сохранена:", metadata_path)


Модель сохранена: /content/drive/MyDrive/credit-scoring-system/models/catboost_all_features_cpu_debug.cbm
Metadata сохранена: /content/drive/MyDrive/credit-scoring-system/models/catboost_all_features_cpu_debug_metadata.json


## 23. Запись результата


In [24]:
current_result = {
    "experiment": EXPERIMENT,
    "notebook": "08_final_model.ipynb",
    "model": "CatBoostClassifier",
    "feature_set": "application + all features",
    "source_tables": (
        "application_train.csv, bureau.csv, bureau_balance.csv, "
        "previous_application.csv, installments_payments.csv, "
        "POS_CASH_balance.csv, credit_card_balance.csv"
    ),
    "device": catboost_device_config["task_type"],
    "n_train": len(train_data),
    "n_holdout": len(holdout_data),
    "n_features": X_train.shape[1],
    "cv_folds": CV_FOLDS,
    "best_iteration": best_iteration,
    "cv_roc_auc": cv_roc_auc,
    "cv_roc_auc_std": cv_roc_auc_std,
    "cv_pr_auc": cv_pr_auc,
    "cv_pr_auc_std": cv_pr_auc_std,
    "holdout_roc_auc": holdout_roc_auc,
    "holdout_pr_auc": holdout_pr_auc,
}

all_results = save_experiment_result(current_result)
display(all_results)


,experiment,notebook,model,feature_set,source_tables,device,n_train,n_holdout,n_features,cv_folds,best_iteration,cv_roc_auc,cv_roc_auc_std,cv_pr_auc,cv_pr_auc_std,holdout_roc_auc,holdout_pr_auc
0,application_logistic,02_application_baseline.ipynb,LogisticRegression,application,application_train.csv,CPU,246008,61503,120,3,NaN,0.744841,0.002434,0.217865,0.005206,NaN,NaN
1,application_catboost,02_application_baseline.ipynb,CatBoostClassifier,application,application_train.csv,GPU,246008,61503,120,3,1000.0,0.754176,0.002317,NaN,NaN,NaN,NaN
2,application_bureau,03_bureau_features.ipynb,CatBoostClassifier,application + bureau,"application_train.csv, bureau.csv, bureau_bala...",GPU,246008,61503,134,3,1000.0,0.758362,0.001620,NaN,NaN,NaN,NaN
3,application_previous,04_previous_application_features.ipynb,CatBoostClassifier,application + previous_application,"application_train.csv, previous_application.csv",GPU,246008,61503,131,3,1000.0,0.760286,0.002017,NaN,NaN,NaN,NaN
4,application_installments,05_installments_features.ipynb,CatBoostClassifier,application + installments,"application_train.csv, installments_payments.csv",GPU,246008,61503,129,3,1000.0,0.759957,0.000499,NaN,NaN,NaN,NaN
5,application_pos_cash,06_pos_cash_features.ipynb,CatBoostClassifier,application + POS_CASH,"application_train.csv, POS_CASH_balance.csv",GPU,246008,61503,127,3,996.0,0.761249,0.001983,NaN,NaN,NaN,NaN
6,application_credit_card,07_credit_card_features.ipynb,CatBoostClassifier,application + credit_card,"application_train.csv, credit_card_balance.csv",GPU,246008,61503,131,3,1000.0,0.758037,0.002157,NaN,NaN,NaN,NaN
7,application_all_features_cpu_debug,08_final_model.ipynb,CatBoostClassifier,application + all features,"application_train.csv, bureau.csv, bureau_bala...",CPU,50000,15000,172,2,150.0,0.758609,0.009406,NaN,NaN,0.765507,0.249836


## Выводы


In [25]:
print("Эксперимент:", EXPERIMENT)
print("Режим:", ACTUAL_RUN_MODE)
print(f"Количество признаков: {X_train.shape[1]}")
print(f"Лучшая итерация: {best_iteration}")
print(f"CV ROC-AUC: {cv_roc_auc:.4f}")
print(f"CV PR-AUC: {cv_pr_auc:.4f}")
print(f"Holdout ROC-AUC: {holdout_roc_auc:.4f}")
print(f"Holdout PR-AUC: {holdout_pr_auc:.4f}")

if IS_DEBUG:
    print(
        "Это технический CPU-debug запуск; метрики не финальные."
    )
else:
    print("Полноценная модель и metadata сохранены в models/.")


Эксперимент: application_all_features_cpu_debug
Режим: cpu_debug
Количество признаков: 172
Лучшая итерация: 150
CV ROC-AUC: 0.7586
CV PR-AUC: nan
Holdout ROC-AUC: 0.7655
Holdout PR-AUC: 0.2498
Это технический CPU-debug запуск; метрики не финальные.
